# 作业1：LangChain 模型工具调用

**实验目标**

1. 安装 `langchain`、`langchain-openai` 和 `openai-agents`。
2. 使用 LangChain 的 `@tool`、`bind_tools`、`AIMessage.tool_calls` 和 `ToolMessage` 完成一次完整工具调用。
3. 分别演示非流式工具参数生成、流式工具参数生成、工具执行和模型汇总。
4. 回答 LangChain 工具调用与 LLM Function Calling 的区别，以及速度受哪些因素影响。

> 说明：本作业使用 DeepSeek 的 OpenAI 兼容接口，模型为 `deepseek-chat`。API Key 不写入 Notebook；运行时优先读取环境变量 `DEEPSEEK_API_KEY`，没有时使用隐藏输入。

## 一、安装依赖

`openai-agent` 通常指 OpenAI Agents SDK，其 PyPI 安装包名是 **`openai-agents`**，导入包名是 `agents`。本实验主体使用 LangChain，安装 OpenAI Agents SDK 是为了满足作业的环境安装要求。

In [1]:
%pip install -qU langchain langchain-openai openai-agents

Note: you may need to restart the kernel to use updated packages.


## 二、导入模块并配置 DeepSeek

使用 `getpass` 可以让 API Key 以隐藏方式输入，不会显示在输出中，也不会保存到 Notebook。

In [2]:
import os
import time
from getpass import getpass
from importlib.metadata import version

import langchain
import langchain_openai
import agents
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

print("langchain 版本：", version("langchain"))
print("langchain-openai 版本：", version("langchain-openai"))
print("openai-agents 版本：", version("openai-agents"))

api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    api_key = getpass("请输入 DeepSeek API Key（输入内容不会显示）：")

model = ChatOpenAI(
    model="deepseek-chat",
    base_url="https://api.deepseek.com",
    api_key=api_key,
    temperature=0,
    max_retries=2,
    timeout=60,
)

print("模型配置完成：DeepSeek / deepseek-chat")

langchain 版本： 1.3.14
langchain-openai 版本： 1.4.1
openai-agents 版本： 0.19.0


模型配置完成：DeepSeek / deepseek-chat


## 三、定义天气工具

`@tool` 会根据函数名、类型注解和 docstring 生成工具的名称、说明和参数 Schema。模型只能**决定调用哪个工具并生成参数**，真正执行 Python 函数的仍然是本地程序。

In [3]:
@tool
def get_weather(location: str) -> str:
    """查询指定城市的天气。location 应为城市名，例如北京、上海、武汉。"""
    if location == "北京":
        return "北京下雪了，明天还是会下雪～"
    if location == "上海":
        return "上海下冰雹了，明天晴天～"
    if location == "武汉":
        return "武汉有雾霾，明天晴天～"
    return f"{location}天气晴朗。"

model_with_tools = model.bind_tools([get_weather])

tools_by_name = {get_weather.name: get_weather}
print("已绑定工具：", list(tools_by_name))

已绑定工具： ['get_weather']


## 四、非流式：让模型选择工具并生成参数

这一轮只请求模型产生 `tool_calls`，尚未执行工具。模型需要为北京、上海、武汉分别生成一次工具调用。

In [4]:
user_query = "北京和上海、武汉的天气怎么样？今天是2026年5月1日，请在拿到工具结果后总结天气。"
messages = [{"role": "user", "content": user_query}]

start = time.perf_counter()
ai_response = model_with_tools.invoke(messages)
select_tools_seconds = time.perf_counter() - start

print(f"模型选择工具耗时：{select_tools_seconds:.3f} 秒")
print(f"工具调用数量：{len(ai_response.tool_calls)}")

for index, tool_call in enumerate(ai_response.tool_calls, start=1):
    print(f"\n第 {index} 个工具调用")
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

assert len(ai_response.tool_calls) == 3, "模型应为三个城市分别生成工具调用。"
assert {call["args"]["location"] for call in ai_response.tool_calls} == {"北京", "上海", "武汉"}

模型选择工具耗时：1.132 秒
工具调用数量：3

第 1 个工具调用
Tool: get_weather
Args: {'location': '北京'}
ID: call_00_yF1SSZWAoy01JNPy7dQT6126

第 2 个工具调用
Tool: get_weather
Args: {'location': '上海'}
ID: call_01_ySsHIqHNb4lBzMcsovLO3820

第 3 个工具调用
Tool: get_weather
Args: {'location': '武汉'}
ID: call_02_MUaMXT8y2s46oV457c367800


## 五、流式：观察工具调用参数的增量生成

流式调用应当针对**原始用户消息单独演示**。不能先把一个尚未执行的 `AIMessage(tool_calls=...)` 加入消息，再请求模型生成第二轮工具调用；那会破坏正常的消息协议。

流式过程中工具参数可能被拆成多段字符串，因此先收集 `tool_call_chunks`，再使用 `AIMessageChunk` 的加法自动合并为完整 `tool_calls`。

In [5]:
start = time.perf_counter()
streamed_response = None

for chunk in model_with_tools.stream([{"role": "user", "content": user_query}]):
    streamed_response = chunk if streamed_response is None else streamed_response + chunk
    for tool_chunk in chunk.tool_call_chunks:
        visible = {
            key: value
            for key, value in tool_chunk.items()
            if value not in (None, "")
        }
        if visible:
            print("收到参数片段：", visible)

stream_seconds = time.perf_counter() - start
print(f"\n流式工具调用生成耗时：{stream_seconds:.3f} 秒")
print("合并后的完整工具调用：")
for tool_call in streamed_response.tool_calls:
    print(tool_call)

收到参数片段： {'name': 'get_weather', 'id': 'call_00_zz9PvSwCGWbRMSnpkhLN0137', 'index': 0, 'type': 'tool_call_chunk'}
收到参数片段： {'args': '{', 'index': 0, 'type': 'tool_call_chunk'}
收到参数片段： {'args': '"', 'index': 0, 'type': 'tool_call_chunk'}
收到参数片段： {'args': 'location', 'index': 0, 'type': 'tool_call_chunk'}
收到参数片段： {'args': '"', 'index': 0, 'type': 'tool_call_chunk'}
收到参数片段： {'args': ': ', 'index': 0, 'type': 'tool_call_chunk'}
收到参数片段： {'args': '"', 'index': 0, 'type': 'tool_call_chunk'}
收到参数片段： {'args': '北京', 'index': 0, 'type': 'tool_call_chunk'}
收到参数片段： {'args': '"', 'index': 0, 'type': 'tool_call_chunk'}
收到参数片段： {'args': '}', 'index': 0, 'type': 'tool_call_chunk'}
收到参数片段： {'name': 'get_weather', 'id': 'call_01_VsVIQHnsHJmIxz9MQkNt8265', 'index': 1, 'type': 'tool_call_chunk'}
收到参数片段： {'args': '{', 'index': 1, 'type': 'tool_call_chunk'}
收到参数片段： {'args': '"', 'index': 1, 'type': 'tool_call_chunk'}
收到参数片段： {'args': 'location', 'index': 1, 'type': 'tool_call_chunk'}
收到参数片段： {'args': '"', 'ind

## 六、执行工具，并把结果放回消息历史

工具返回值必须作为 `ToolMessage` 放回消息历史，而且 `tool_call_id` 必须和模型生成的调用 ID 对应。LangChain 的 `BaseTool.invoke(tool_call)` 会完成这个封装。

In [6]:
messages.append(ai_response)

tool_start = time.perf_counter()
for tool_call in ai_response.tool_calls:
    selected_tool = tools_by_name[tool_call["name"]]
    tool_result = selected_tool.invoke(tool_call)
    assert isinstance(tool_result, ToolMessage)
    messages.append(tool_result)
    print(f"{tool_call['args']['location']}：{tool_result.content}")
tool_seconds = time.perf_counter() - tool_start

print(f"本地工具执行总耗时：{tool_seconds:.6f} 秒")

北京：北京下雪了，明天还是会下雪～
上海：上海下冰雹了，明天晴天～
武汉：武汉有雾霾，明天晴天～
本地工具执行总耗时：0.003143 秒


## 七、模型根据工具结果生成最终总结

In [7]:
start = time.perf_counter()
final_response = model_with_tools.invoke(messages)
summary_seconds = time.perf_counter() - start

print(f"最终汇总耗时：{summary_seconds:.3f} 秒")
print("\n最终回答：")
print(final_response.text)

assert final_response.text.strip(), "最终回答不应为空。"

最终汇总耗时：2.659 秒

最终回答：
好的，以下是今天（2026年5月1日）北京、上海、武汉三地的天气情况总结：

---

### 🌤️ 今日天气总结（2026年5月1日）

| 城市 | 今日天气 | 明日天气 |
|:---:|:-------:|:-------:|
| **北京** 🏯 | ❄️ **下雪** | ❄️ 继续下雪 |
| **上海** 🌃 | 🧊 **下冰雹** | ☀️ 晴天 |
| **武汉** 🏙️ | 🌫️ **雾霾** | ☀️ 晴天 |

---

**详细说明：**

1. **北京**：今天竟然在5月飘雪了！❄️ 而且明天还会继续下雪，比较反常，出门注意保暖和防滑。
2. **上海**：今天遭遇冰雹天气🧊，比较罕见，注意安全。不过好消息是明天就会转晴啦！☀️
3. **武汉**：今天有雾霾🌫️，能见度可能不太好，建议佩戴口罩。明天也会转为晴天☀️。

总体来看，今天三地天气都不太寻常，但上海和武汉明天都会迎来好天气，北京则还要再等一天哦～


## 八、耗时观察

下面的耗时是本次运行的实测值。工具函数只是本地字符串判断，耗时很小；主要时间通常消耗在两次远程模型请求上。

In [8]:
print(f"模型选择工具：{select_tools_seconds:.3f} 秒")
print(f"本地执行工具：{tool_seconds:.6f} 秒")
print(f"模型最终汇总：{summary_seconds:.3f} 秒")
print(f"主流程总耗时：{select_tools_seconds + tool_seconds + summary_seconds:.3f} 秒（不含独立的流式演示）")

模型选择工具：1.132 秒
本地执行工具：0.003143 秒
模型最终汇总：2.659 秒
主流程总耗时：3.794 秒（不含独立的流式演示）


## 九、问题回答

### 1. LangChain 工具调用和 LLM Function Call 有什么区别？

二者不是互相替代的同一层概念：

- **LLM Function Calling（底层模型/API 能力）**：把函数名、描述和参数 Schema 发给模型，模型返回“调用哪个函数、参数是什么”的结构化数据。模型本身通常**不会执行函数**。不同厂商的字段和协议可能不同。
- **LangChain 工具调用（上层抽象与编排）**：LangChain 用 `@tool`/`BaseTool` 封装 Python 函数，用 `bind_tools()` 把工具转换为模型供应商需要的 Schema，并把不同模型返回值统一成 `AIMessage.tool_calls`；工具执行结果统一封装为 `ToolMessage`。如果使用 `create_agent`，LangChain 还能自动循环完成“模型选择工具 → 执行工具 → 把结果返回模型 → 生成最终答案”。

因此，本实验中的 `model.bind_tools([get_weather])` **只负责绑定和规范化**，不会自动执行 `get_weather`；代码仍需遍历 `ai_response.tool_calls` 并调用工具。可以把它理解为：**Function Calling 是发动机提供的能力，LangChain 是统一接口和工作流框架。**

### 2. LangChain 工具调用的速度受到什么影响？

总耗时可以近似拆成：

> 总耗时 ≈ 模型选择工具耗时 + 工具执行耗时 + 模型汇总耗时 + 框架/序列化开销

主要影响因素如下：

1. **模型推理速度**：模型大小、服务负载、是否为推理模型、输出 token 数都会影响首轮选工具和末轮总结。
2. **网络与服务端延迟**：客户端到模型 API 的网络 RTT、跨地域访问、限流、排队和重试，通常比 LangChain 本身的 Python 开销更明显。
3. **模型调用轮数**：本实验至少有两次 LLM 请求；Agent 若多轮规划、反思或反复调工具，会线性甚至更高地增加耗时。
4. **上下文长度与工具数量**：历史消息越长、工具越多、每个工具描述和 JSON Schema 越复杂，输入 token 越多，模型选择工具越慢。
5. **工具自身耗时**：本地计算通常很快；数据库、搜索、文件处理、外部 HTTP API 会显著增加延迟。
6. **串行还是并行**：多个互不依赖的工具串行执行时总耗时近似相加；并行执行时更接近最慢工具的耗时。本例模型一次返回了三个工具调用，但示例代码是串行执行。
7. **流式输出**：流式通常能缩短“用户看到第一个片段”的时间，但不保证降低完整响应的总耗时；还会有分块合并开销。
8. **超时、重试、回调与追踪**：失败重试、LangSmith tracing、日志回调、复杂中间件都会增加一定开销。

本实验的计时结果能直观看出：`get_weather` 只是本地条件判断，执行接近瞬时；速度瓶颈主要是 DeepSeek 的两次远程模型调用，而不是 `@tool` 或 `bind_tools()` 本身。

## 十、实验结论

本实验完成了完整工具调用闭环：

1. 使用 `@tool` 将普通 Python 函数封装为 LangChain Tool；
2. 使用 `bind_tools()` 向 DeepSeek 暴露工具 Schema；
3. 模型为三个城市生成结构化工具调用；
4. Python 程序执行工具并返回 `ToolMessage`；
5. 模型基于工具结果生成天气总结；
6. 通过分阶段计时确认，主要延迟来自远程模型推理和网络，而非 LangChain 的工具封装。